# Homework — SOLUTIONS — Special Methods, Access Modifiers, Inheritance & Files

**Theme: Gym Membership System**

This homework mirrors what we did in class, applied to a new scenario, so you can check your own
understanding independently. Sections build on each other — do them in order.

| Section | Topics |
|---|---|
| 🟢 Core | `__init__`, access modifiers, `__str__`, comparison operators |
| 🟡 Stretch | Inheritance, `super()`, overriding, composition |
| 🔴 Challenge | File I/O, iteration, an extra special method |

As in class: each class lives in one cell. If a later exercise asks you to add a method to a
class you already wrote, scroll up, add it there, and re-run.


## 🟢 Core — Exercise 1: The `Member` class

In [1]:
class Member:
    _id_counter = 0   # class attribute: shared counter for auto-generating IDs

    def __init__(self, name, email, age):
        self.name = name
        self.email = email
        self._age = age                      # protected
        Member._id_counter += 1
        self.__member_id = Member._id_counter  # private

    def get_member_id(self):
        return self.__member_id

    def get_age(self):
        return self._age

    def __str__(self):
        return f"Member #{self.get_member_id()}: {self.name} ({self.email})"

    def __eq__(self, other):
        return self.get_member_id() == other.get_member_id()

    def __lt__(self, other):
        return self._age < other._age


In [2]:
alice = Member("Alice", "alice@mail.com", 28)
bob = Member("Bob", "bob@mail.com", 34)

print(alice)
print(bob)
print(alice.get_member_id())
print(alice < bob)
print(alice == bob)


Member #1: Alice (alice@mail.com)
Member #2: Bob (bob@mail.com)
1
True
False


## 🟡 Stretch — Exercise 2: `BasicMember` and `PremiumMember`

In [3]:
class BasicMember(Member):
    def __init__(self, name, email, age):
        super().__init__(name, email, age)

    def get_monthly_fee(self):
        return 200


class PremiumMember(Member):
    def __init__(self, name, email, age, personal_trainer=None):
        super().__init__(name, email, age)
        self.personal_trainer = personal_trainer

    def get_monthly_fee(self):
        fee = 500
        if self.personal_trainer:
            fee += 150
        return fee


In [4]:
basic = BasicMember("Chris", "chris@mail.com", 22)
premium = PremiumMember("Dana", "dana@mail.com", 40, "Coach Kim")

print(basic.get_monthly_fee())
print(premium.get_monthly_fee())
print(basic)


200
650
Member #3: Chris (chris@mail.com)


## 🟡 Stretch / 🔴 Challenge — Exercise 3: `Gym` class (full version)

In [5]:
class Gym:
    def __init__(self, name):
        self.name = name
        self.members = []

    def add_member(self, member):
        self.members.append(member)

    def total_monthly_revenue(self):
        return sum(member.get_monthly_fee() for member in self.members)

    def find_by_email(self, email):
        for member in self.members:
            if member.email == email:
                return member
        return None

    def __len__(self):
        return len(self.members)

    def __getitem__(self, index):
        return self.members[index]

    # ---- Challenge methods ----

    def __iter__(self):
        self._iter_index = 0
        return self

    def __next__(self):
        if self._iter_index >= len(self.members):
            raise StopIteration
        member = self.members[self._iter_index]
        self._iter_index += 1
        return member

    def __bool__(self):
        return len(self) > 0

    def save_to_file(self, filename):
        with open(filename, "w") as f:
            for member in self.members:
                f.write(f"{member.name},{member.email},{member.get_age()}\n")

    @classmethod
    def load_from_file(cls, filename, name="Loaded Gym"):
        gym = cls(name)
        with open(filename, "r") as f:
            for line in f:
                member_name, email, age = line.strip().split(",")
                gym.add_member(Member(member_name, email, int(age)))
        return gym


In [6]:
gym = Gym("PowerGym")
gym.add_member(basic)
gym.add_member(premium)

print(len(gym))
print(gym.total_monthly_revenue())
print(gym.find_by_email("dana@mail.com"))


2
850
Member #4: Dana (dana@mail.com)


## 🔴 Challenge — Exercise 4: Iteration and `__bool__`

In [7]:
for member in gym:
    print(member)

empty_gym = Gym("New Gym")
print(bool(gym))
print(bool(empty_gym))


Member #3: Chris (chris@mail.com)
Member #4: Dana (dana@mail.com)
True
False


## 🔴 Challenge — Exercise 5: Save / load

In [8]:
gym.save_to_file("members.txt")
reloaded_gym = Gym.load_from_file("members.txt", "Reloaded Gym")
for member in reloaded_gym:
    print(member)


Member #5: Chris (chris@mail.com)
Member #6: Dana (dana@mail.com)
